# Pipeline KDD — CAR/SICAR

Sistematização de Ciência de Dados II. Ver `plano_trabalho.md` e `RELATORIO.md` para contexto.

## Etapa 1 — Seleção do Dataset

- **Fonte:** Portal oficial do SICAR (consultapublica.car.gov.br/publico/estados/downloads),
  camada "Área do Imóvel", estado da **Bahia (BA)** — baixado manualmente (o download exige
  captcha, não dá pra automatizar) e convertido de `.dbf` para `.csv` com `src/leitor_dbf.py`
  (leitor próprio do formato .dbf, sem geopandas/fiona).
- **Volume:** 1.313.653 linhas (bem acima do mínimo de 100 mil exigido) × 12 colunas — CSV de
  ~184 MB.
- **Colunas:** `cod_tema`, `nom_tema` (constantes — sempre "AREA_IMOVEL"/"Area do Imovel" nesta
  camada), `cod_imovel` (identificador único do imóvel no SICAR), `mod_fiscal` (tamanho em
  módulos fiscais), `num_area` (área total declarada), `ind_status` (situação do cadastro —
  ex. "AT" ativo — **alvo da Etapa 4**), `ind_tipo` (tipo de imóvel, ex. "IRU" rural),
  `des_condic` (descrição textual da condição, ex. "Aguardando analise"), `municipio`,
  `cod_estado`, `dat_criaca`/`dat_atuali` (datas de criação/atualização do registro, formato
  `DD/MM/AAAA` — vêm como string, precisam de `to_date(..., 'dd/MM/yyyy')` na Etapa 2).
- **Justificativa:** CAR/SICAR é o cadastro nacional de imóveis rurais, público e tabular por
  natureza; a Bahia sozinha já ultrapassa em mais de 13x o volume mínimo exigido pelo trabalho,
  e o tema é relevante para o país e a dificuldade de manter seu cadastro rural atualizado devido a sua dimensão territorial.
  `cod_tema`/`nom_tema` não agregam informação (valor único) e serão descartadas na Etapa 2.

In [ ]:
# No Windows, o Spark às vezes escolhe o Python errado pra abrir o processo
# "worker" (viramos vítimas disso no diagnóstico do ambiente: dava
# "java.net.SocketException: Connection reset" porque ele tentava usar um
# outro Python instalado na máquina, não este). Forçar PYSPARK_PYTHON pro
# mesmo interpretador que está rodando o notebook resolve.
import sys, os
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("car-sicar-kdd")
    .master("local[*]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .getOrCreate()
)
spark

In [ ]:
df = spark.read.csv("../data/raw/AREA_IMOVEL_1.csv", header=True, inferSchema=True)
df.printSchema()
df.count()

## Etapa 2 — Ingestão e Pré-processamento com Spark

Tratamento de nulos, tipos, duplicatas, outliers e engenharia de atributos.

**Achados principais:**

- `cod_tema` e `nom_tema` são constantes em todo o dataset (só existe "AREA_IMOVEL" /
  "Area do Imovel") — não carregam informação nenhuma, foram descartadas.
- 33 grupos de `cod_imovel` duplicado (66 linhas no total). Investigando um exemplo,
  o padrão ficou claro pela coluna `des_condic`: cada duplicidade é uma linha com
  `ind_status = "AT"` (ativa) mais uma ou mais linhas com status de cancelamento e
  `des_condic` explicando o motivo — no caso mais comum, "Cancelado por duplicidade".
  Verificado que o padrão se repete em todos os 33 grupos, não só no exemplo. Resolvido
  mantendo a linha `"AT"` de cada grupo (ou a mais recente por `dat_atuali`, quando não
  há nenhuma ativa) — de 1.313.653 linhas para 1.313.620, batendo exatamente com o
  número de `cod_imovel` distintos.
- 8 nulos em `dat_atuali` (nenhum nulo nas demais colunas), preenchidos com o valor de
  `dat_criaca` (assume-se que um imóvel nunca atualizado manteve a data de criação como
  última atualização).
- `dat_criaca` e `dat_atuali` vieram como texto no formato `DD/MM/AAAA` — convertidas
  para o tipo `date` do Spark.
- Não foram encontrados outliers óbvios de área (`num_area`) ou módulo fiscal
  (`mod_fiscal`) negativos ou zerados.
- `ind_status` é fortemente desbalanceado (~99,4% `AT`) — relevante para a Etapa 4
  (modelagem preditiva), vai precisar de tratamento de desbalanceamento (ex. pesos por
  classe, undersampling/oversampling, ou métricas que não sejam só acurácia).


In [ ]:
from pyspark.sql import functions as F

# 1) cod_tema/nom_tema são mesmo constantes (sem informação nenhuma)?
df.select("cod_tema", "nom_tema").distinct().show()

# 2) quantos nulos tem em cada coluna?
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).show()

# 3) cod_imovel é único, ou tem imóvel repetido?
total = df.count()
distintos = df.select("cod_imovel").distinct().count()
print(f"total de linhas: {total} | cod_imovel distintos: {distintos}")

# 4) quais valores existem em ind_status e ind_tipo, e quão comum é cada um?
df.groupBy("ind_status").count().orderBy(F.desc("count")).show()
df.groupBy("ind_tipo").count().orderBy(F.desc("count")).show()

In [ ]:
# 5) investigar os cod_imovel duplicados
duplicados = df.groupBy("cod_imovel").count().filter("count > 1")
duplicados.orderBy(F.desc("count")).show(10, truncate=False)

duplicados_lista = [row["cod_imovel"] for row in duplicados.collect()]
df.filter(df.cod_imovel.isin(duplicados_lista)) \
  .select("cod_imovel", "ind_status", "des_condic", "dat_atuali") \
  .orderBy("cod_imovel") \
  .show(len(duplicados_lista) * 2, truncate=False)

# 6) checar outliers óbvios de área / módulo fiscal (negativos ou zerados)
df.select("mod_fiscal", "num_area").describe().show()
df.filter((F.col("num_area") <= 0) | (F.col("mod_fiscal") <= 0)).count()


In [ ]:
from pyspark.sql import Window

# cada grupo duplicado tem uma linha "AT" (ativa) + uma ou mais canceladas por
# duplicidade: ficamos com a linha AT quando existe, senão a mais recente por dat_atuali
janela = Window.partitionBy("cod_imovel").orderBy(
    F.when(F.col("ind_status") == "AT", 0).otherwise(1),
    F.desc("dat_atuali")
)

df_dedup = (
    df.withColumn("rn", F.row_number().over(janela))
      .filter(F.col("rn") == 1)
      .drop("rn")
)

print(f"antes: {df.count()} | depois: {df_dedup.count()}")
print(f"cod_imovel distintos depois: {df_dedup.select('cod_imovel').distinct().count()}")


In [ ]:
# descartar colunas constantes, preencher nulos e converter datas
# (o coalesce precisa acontecer ANTES da conversão pra date, enquanto as duas
# colunas ainda são texto)
df_limpo = (
    df_dedup
    .drop("cod_tema", "nom_tema")
    .withColumn("dat_atuali", F.coalesce(F.col("dat_atuali"), F.col("dat_criaca")))
    .withColumn("dat_criaca", F.to_date(F.col("dat_criaca"), "dd/MM/yyyy"))
    .withColumn("dat_atuali", F.to_date(F.col("dat_atuali"), "dd/MM/yyyy"))
)

df_limpo.printSchema()
df_limpo.select("dat_criaca", "dat_atuali").show(5)

# checagem final: não deve sobrar nenhum nulo
df_limpo.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_limpo.columns]).show()


## Etapa 3 — Análise Exploratória com Spark SQL

Registrar o DataFrame como view e responder pelo menos 5 perguntas de negócio.

In [ ]:
# df.createOrReplaceTempView("imoveis")
# spark.sql("SELECT ... FROM imoveis ...").show()

### Pergunta 1


### Pergunta 2


### Pergunta 3


### Pergunta 4


### Pergunta 5


## Etapa 4 — Modelagem Preditiva

Problema (classificação/regressão), 2+ modelos de famílias diferentes (1 ensemble), métricas e validação.

## Etapa 5 — Modelagem Descritiva

Clusterização (ex. K-Means), escolha do k (cotovelo/silhueta) e interpretação dos perfis.

## Etapa 6 — Interpretação e Conclusões (KDD)

Conhecimento descoberto, decisões possíveis, limitações e próximos passos.